Сначала код группирует процессоры по шести факторам, которые должны совпадать точно. 
Затем ищет внутри этих групп 20 строк с минимальным разбросом по частоте и возрасту. 
Чтобы сравнить ГГц и годы, каждый размах делится на стандартное отклонение своего признака по всему набору без пропусков. 
При отборе cpu_mark не используется.

In [2]:
import pandas as pd

df = pd.read_csv("final5.csv")
data = df.loc[df["specs_missing_count"].eq(0)]

fixed = [
    "manufacturer_Intel", "manufacturer_Other", "category_Laptop",
    "cores", "tdp_w", "l3_cache_mb",
]
clock, age = "base_clock_ghz", "years_ago"
size = 20
clock_scale, age_scale = data[[clock, age]].std()

best_score = float("inf")
extra = None

for _, group in data.groupby(fixed):
    if len(group) < size:
        continue

    clocks = sorted(group[clock].unique())
    for i, low in enumerate(clocks):
        for high in clocks[i:]:
            window = group.loc[group[clock].between(low, high)].sort_values(age)
            for start in range(len(window) - size + 1):
                candidate = window.iloc[start:start + size]
                clock_span = (candidate[clock].max() - candidate[clock].min()) / clock_scale
                age_span = (candidate[age].max() - candidate[age].min()) / age_scale
                score = max(clock_span, age_span)

                if score < best_score:
                    best_score = score
                    extra = candidate

if extra is None:
    print(f"Группа из {size} строк с совпадающими факторами не найдена.")
else:
    extra.sort_index().to_csv("cpu_extra_sample.csv", index=False)
    print(f"Сохранено {len(extra)} строк в cpu_extra_sample.csv")
    display(extra[fixed + [clock, age]].agg(["min", "max"]))

Сохранено 20 строк в cpu_extra_sample.csv


,manufacturer_Intel,manufacturer_Other,category_Laptop,cores,tdp_w,l3_cache_mb,base_clock_ghz,years_ago
min,1,0,1,2,35.0,3.0,2.1,15
max,1,0,1,2,35.0,3.0,2.5,16


In [3]:
import numpy as np
import pandas as pd

extra = pd.read_csv("cpu_extra_sample.csv")
y = np.log(extra["cpu_mark"])

df_0 = len(y) - 1
S0_sq = ((y - y.mean()) ** 2).sum() / df_0

print(f"Число процессоров = {len(y)}")
print(f"Степени свободы = {df_0}")
print(f"S₀² = {S0_sq:.6f}")

Число процессоров = 20
Степени свободы = 19
S₀² = 0.080688


In [5]:
C = pd.read_csv("error_matrix.csv")

In [6]:
import numpy as np
import pandas as pd

cov_B = S0_sq * C
variances = pd.Series(np.diag(cov_B), index=cov_B.index)

coefficient_errors = pd.DataFrame({
    "Дисперсия оценки": variances,
    "Стандартная ошибка": np.sqrt(variances),
})

display(cov_B)
display(coefficient_errors)

,const,log1p_cores,base_clock_ghz,log1p_tdp,log1p_l3,age_centered,manufacturer_Intel,manufacturer_Other,category_Laptop
0,5.911210e-05,-6.730838e-21,6.696728e-20,-3.710796e-20,-2.261242e-20,2.109556e-21,-2.013358e-20,-1.834490e-20,1.645142e-20
1,-7.399028e-21,1.858577e-04,3.182062e-05,-4.238551e-05,-1.252914e-04,4.443488e-05,3.391946e-06,-3.236622e-06,1.656307e-05
2,6.564760e-20,3.182062e-05,1.739850e-04,-1.139685e-04,-3.835766e-05,4.511456e-05,-1.618631e-05,-4.021661e-06,3.271214e-05
3,-3.874690e-20,-4.238551e-05,-1.139685e-04,2.002713e-04,-1.897493e-05,-7.180197e-05,1.315812e-05,-1.687801e-05,4.062606e-05
4,-2.133658e-20,-1.252914e-04,-3.835766e-05,-1.897493e-05,2.062912e-04,2.419681e-05,1.125023e-05,4.524878e-05,-3.162555e-05
5,2.266302e-21,4.443488e-05,4.511456e-05,-7.180197e-05,2.419681e-05,1.136274e-04,1.944274e-05,1.218737e-05,-3.885399e-06
6,-1.967018e-20,3.391946e-06,-1.618631e-05,1.315812e-05,1.125023e-05,1.944274e-05,7.741054e-05,2.768815e-05,-1.148679e-05
7,-1.752956e-20,-3.236622e-06,-4.021661e-06,-1.687801e-05,4.524878e-05,1.218737e-05,2.768815e-05,8.554853e-05,-2.021065e-05
8,1.226602e-20,1.656307e-05,3.271214e-05,4.062606e-05,-3.162555e-05,-3.885399e-06,-1.148679e-05,-2.021065e-05,1.037475e-04


,Дисперсия оценки,Стандартная ошибка
0,0.000059,0.007688
1,0.000186,0.013633
2,0.000174,0.013190
3,0.000200,0.014152
4,0.000206,0.014363
5,0.000114,0.010660
6,0.000077,0.008798
7,0.000086,0.009249
8,0.000104,0.010186


In [8]:
import numpy as np
import pandas as pd
from scipy.stats import t
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

raw = pd.read_csv("final5.csv")
development, _ = train_test_split(
    raw, test_size=300, random_state=42, stratify=raw["category_Laptop"]
)
development = development.sort_index()

features = development.assign(
    log1p_cores=np.log1p(development["cores"]),
    log1p_tdp=np.log1p(development["tdp_w"]),
    log1p_l3=np.log1p(development["l3_cache_mb"]),
    age_centered=development["years_ago"] - development["years_ago"].mean(),
)[[
    "log1p_cores", "base_clock_ghz", "log1p_tdp", "log1p_l3",
    "age_centered", "manufacturer_Intel", "manufacturer_Other", "category_Laptop",
]]

Z = StandardScaler().fit_transform(features)
X = np.column_stack([np.ones(len(Z)), Z])
Y = np.log(development["cpu_mark"])
B = np.linalg.lstsq(X, Y, rcond=None)[0]

extra = pd.read_csv("cpu_extra_sample.csv")
y_extra = np.log(extra["cpu_mark"])
S0_sq = y_extra.var(ddof=1)
df_0 = len(y_extra) - 1
cov_B = S0_sq * np.linalg.inv(X.T @ X)

nearest = np.argmin((Z ** 2).sum(axis=1))
x_star = X[nearest]

y_hat_star = x_star @ B
var_y_hat_star = x_star @ cov_B @ x_star
margin = t.ppf(0.975, df=df_0) * np.sqrt(var_y_hat_star)

print(f"Индекс процессора в исходных данных = {development.index[nearest]}")
print(f"Прогноз ln(CPU Mark) = {y_hat_star:.6f}")
print(f"Дисперсия прогноза среднего = {var_y_hat_star:.6f}")
print(f"95%-й ДИ = [{y_hat_star - margin:.6f}, {y_hat_star + margin:.6f}]")

Индекс процессора в исходных данных = 1431
Прогноз ln(CPU Mark) = 7.274314
Дисперсия прогноза среднего = 0.000201
95%-й ДИ = [7.244612, 7.304016]
